## Imports

In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
try:
    from catboost import CatBoostClassifier
except ModuleNotFoundError:
    ! pip install catboost
    from catboost import CatBoostClassifier
try:
    import xgboost as xgb
except ModuleNotFoundError:
    ! pip install xgboost
    import xgboost as xgb
try:
    from sklearn.preprocessing import StandardScaler
except ModuleNotFoundError:
    ! pip install scikit-learn as sklearn
    from sklearn.preprocessing import StandardScaler # for feature scaling such that i can do log reg
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split #to split the data into train and test
from sklearn.linear_model import LogisticRegression #model
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score #evaluation

### Functions

In [2]:
# denote df
def mark_df(flt_prop_row):
    if flt_prop_row <= 0.6:
        return 'train'
    elif flt_prop_row <= 0.8:
        return 'valid'
    else:
        return 'test'

## Constants

In [3]:
str_project = os.getcwd().split('\\')[5]
print(f'Project: {str_project}')
str_task = os.getcwd().split('\\')[6]
print(f'Task: {str_task}')
str_subtask = os.getcwd().split('\\')[7]
print(f'Subtask: {str_subtask}')

str_dirname_output = './output'

Project: 06_ml_in_python
Task: 01_home_credit_model
Subtask: 04_model_building


In [4]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

## Read in train.csv to Df

In [5]:
str_filename = 'train.csv'
str_local_path = f'./input/{str_filename}'
df = pd.read_csv(str_local_path)
df.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,396902,0,Cash loans,F,Y,Y,0,121500,835380,40320.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0
1,112096,0,Cash loans,F,N,Y,0,202500,516069,26478.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,6.0
2,285821,1,Cash loans,M,Y,Y,1,180000,284400,22469.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
3,376901,0,Cash loans,F,N,Y,0,90000,265536,13685.0,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,1.0
4,325138,0,Cash loans,F,Y,Y,0,94500,755190,30078.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


## Imputation

In [6]:
# % of missing vals in each column
missing_vals = df.isnull().mean() * 100
# show
missing_cols = missing_vals[missing_vals > 0]
missing_cols = missing_cols.sort_values(ascending=False)
missing_cols

COMMONAREA_MEDI             69.7816
COMMONAREA_AVG              69.7816
COMMONAREA_MODE             69.7816
NONLIVINGAPARTMENTS_MEDI    69.3440
NONLIVINGAPARTMENTS_MODE    69.3440
                             ...   
OBS_30_CNT_SOCIAL_CIRCLE     0.3000
EXT_SOURCE_2                 0.2320
AMT_GOODS_PRICE              0.1056
AMT_ANNUITY                  0.0024
CNT_FAM_MEMBERS              0.0008
Length: 66, dtype: float64

In [10]:
# identify columns to remove with a 50% threshold
cols_to_remove = missing_cols[missing_cols > 50].index.tolist()
#show
cols_to_remove

['COMMONAREA_MEDI',
 'COMMONAREA_AVG',
 'COMMONAREA_MODE',
 'NONLIVINGAPARTMENTS_MEDI',
 'NONLIVINGAPARTMENTS_MODE',
 'NONLIVINGAPARTMENTS_AVG',
 'FONDKAPREMONT_MODE',
 'LIVINGAPARTMENTS_MODE',
 'LIVINGAPARTMENTS_MEDI',
 'LIVINGAPARTMENTS_AVG',
 'FLOORSMIN_MODE',
 'FLOORSMIN_MEDI',
 'FLOORSMIN_AVG',
 'YEARS_BUILD_MODE',
 'YEARS_BUILD_MEDI',
 'YEARS_BUILD_AVG',
 'OWN_CAR_AGE',
 'LANDAREA_AVG',
 'LANDAREA_MEDI',
 'LANDAREA_MODE',
 'BASEMENTAREA_MEDI',
 'BASEMENTAREA_AVG',
 'BASEMENTAREA_MODE',
 'EXT_SOURCE_1',
 'NONLIVINGAREA_MEDI',
 'NONLIVINGAREA_MODE',
 'NONLIVINGAREA_AVG',
 'ELEVATORS_MEDI',
 'ELEVATORS_MODE',
 'ELEVATORS_AVG',
 'WALLSMATERIAL_MODE',
 'APARTMENTS_MODE',
 'APARTMENTS_MEDI',
 'APARTMENTS_AVG',
 'ENTRANCES_MODE',
 'ENTRANCES_AVG',
 'ENTRANCES_MEDI',
 'LIVINGAREA_MEDI',
 'LIVINGAREA_MODE',
 'LIVINGAREA_AVG',
 'HOUSETYPE_MODE']

In [11]:
#remove those cols
df_clean = df.drop(columns=cols_to_remove)
#list of num cols and cat cols
numerical_cols = df_clean.select_dtypes(include=['float64','int64']).columns
categorical_cols = df_clean.select_dtypes(include=['object']).columns
#impute num cols with mean and cat cols with mode
df_clean[numerical_cols] = df_clean[numerical_cols].fillna(df_clean[numerical_cols].mean())
for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
#checking that the imp worked
missing_vals_post = df_clean.isnull().mean() * 100
#show
missing_vals_post[missing_vals_post > 0]

Series([], dtype: float64)

## Encoding Cat Cols

In [12]:
#use one-hot encoding on categorical_cols from above
# df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)
#show
df_encoded = df_clean

In [13]:
#check to make sure there are no 'object' dtypes
print(df_encoded.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125000 entries, 0 to 124999
Data columns (total 81 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   SK_ID_CURR                    125000 non-null  int64  
 1   TARGET                        125000 non-null  int64  
 2   NAME_CONTRACT_TYPE            125000 non-null  object 
 3   CODE_GENDER                   125000 non-null  object 
 4   FLAG_OWN_CAR                  125000 non-null  object 
 5   FLAG_OWN_REALTY               125000 non-null  object 
 6   CNT_CHILDREN                  125000 non-null  int64  
 7   AMT_INCOME_TOTAL              125000 non-null  int64  
 8   AMT_CREDIT                    125000 non-null  int64  
 9   AMT_ANNUITY                   125000 non-null  float64
 10  AMT_GOODS_PRICE               125000 non-null  float64
 11  NAME_TYPE_SUITE               125000 non-null  object 
 12  NAME_INCOME_TYPE              125000 non-nul

## Feature Scaling

In [14]:
#exclude TARGET from num cols
numerical_cols = df_encoded.drop(columns=['TARGET']).select_dtypes(include=['float64','int64']).columns
#initialize scaler
scaler = StandardScaler()
# apply standardization to num cols
df_encoded[numerical_cols] = scaler.fit_transform(df_encoded[numerical_cols])
#show
df_encoded.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,1.152196,0,Cash loans,F,Y,Y,-0.578560,-0.138764,0.584766,0.911707,...,-0.092743,-0.026239,-0.020981,-0.019187,-0.082944,-6.680620e-02,-1.801719e-01,-3.127393e-01,-4.640186e-01,1.203308e+00
1,-1.610966,0,Cash loans,F,N,Y,-0.578560,0.096055,-0.206473,-0.042928,...,-0.092743,-0.026239,-0.020981,-0.019187,-0.082944,-6.680620e-02,-1.801719e-01,-3.127393e-01,-4.640186e-01,2.351662e+00
2,0.074498,1,Cash loans,M,Y,Y,0.806695,0.030827,-0.780539,-0.319415,...,-0.092743,-0.026239,-0.020981,-0.019187,-0.082944,-6.680620e-02,-1.801719e-01,-3.127393e-01,-4.640186e-01,6.291308e-01
3,0.958148,0,Cash loans,F,N,Y,-0.578560,-0.230083,-0.827284,-0.925217,...,-0.092743,-0.026239,-0.020981,-0.019187,-0.082944,-6.680620e-02,-1.801719e-01,-3.127393e-01,1.306120e+00,-5.192232e-01
4,0.455948,0,Cash loans,F,Y,Y,-0.578560,-0.217037,0.386059,0.205352,...,-0.092743,-0.026239,-0.020981,-0.019187,0.000000,-1.650598e-17,7.244498e-17,6.511397e-17,9.826246e-17,1.274929e-16


### Splitting Aaron's way

In [15]:
# create column
int_nrows = df_encoded.shape[0]
df_encoded['int_row'] = list(range(1, int_nrows+1))
# divide by n rows
df_encoded['flt_prop_row'] = df_encoded['int_row'] / int_nrows

# drop
df_encoded.drop('int_row', axis=1, inplace=True)

# mark the df
df_encoded['data_set'] = df_encoded['flt_prop_row'].apply(mark_df)

### Write dfs

In [16]:
list_str_df = [
    'train',
    'valid',
    'test',
]
for str_df in tqdm(list_str_df):
    # subset
    df_tmp = df_encoded[df_encoded['data_set'] == str_df].copy()
    # write to file
    str_filename = f'df_{str_df}.gzip'
    str_local_path = f'{str_dirname_output}/{str_filename}'
    df_tmp.to_parquet(str_local_path, compression='gzip')

100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:02<00:00,  1.20it/s]


### Random data split

In [ ]:
#def features (X) and target (Y)
X = df_encoded.drop(columns=['TARGET'])
Y = df_encoded['TARGET']
#split the data
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
#show
print(X_train.shape, X_test.shape, Y_train.shape, Y_test.shape)

## Build and Eval Logistic Regression Model

### Training

In [ ]:
#initialize model
model = LogisticRegression(max_iter=1000, random_state=42)
#train model
model.fit(X_train, Y_train)

### Evaluation

In [ ]:
#predict probabilities
Y_pred_prob = model.predict_proba(X_test)[:, 1]
#calc ROC AUC score
roc_auc = roc_auc_score(Y_test, Y_pred_prob)
#classification report
Y_pred = model.predict(X_test)
report = classification_report(Y_test, Y_pred, output_dict=True)
df_report = pd.DataFrame(report).transpose()
#show
df_report, roc_auc

## Build and Eval CatBoostClassifier Model

### Pre-Requisites

In [ ]:
df_clean = df.drop(columns=cols_to_remove)
numerical_cols = df_clean.select_dtypes(include=['float64','int64']).columns
categorical_cols = df_clean.select_dtypes(include=['object']).columns
df_clean[numerical_cols] = df_clean[numerical_cols].fillna(df_clean[numerical_cols].mean())
for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
categorical_cols = categorical_cols.tolist()
missing_vals_post = df_clean.isnull().mean() * 100
missing_vals_post[missing_vals_post > 0]

### Data Splitting

In [ ]:
X = df_clean.drop(columns=['TARGET'])
Y = df_clean['TARGET']
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape, Y_train.shape, Y_test.shape)

### Training

#### Verify Cat Cols w/ X_train

In [ ]:
#verify cat cols in X_train
missing_cat_cols = [col for col in categorical_cols if col not in X_train.columns]
print("Missing categorical columns:", missing_cat_cols)

In [ ]:
#take out the missing cat cols from categorical_cols list
categorical_cols = [col for col in categorical_cols if col in X_train.columns]

#### Model_1

In [ ]:
model_1 = CatBoostClassifier(iterations=1000, learning_rate=0.1, depth=6, random_seed=42, verbose=100)
model_1.fit(X_train, Y_train, cat_features=categorical_cols)

### Evaluation_1

In [ ]:
Y_pred_prob = model_1.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(Y_test, Y_pred_prob)
Y_pred = model_1.predict(X_test)
report = classification_report(Y_test, Y_pred, output_dict=True)
df_report = pd.DataFrame(report).transpose()
df_report, roc_auc

### Model_2 (Gen xii Tune)

In [ ]:
model_2 = CatBoostClassifier(
    task_type='CPU',
    nan_mode='Min',
    random_state=42,
    iterations=100,
    learning_rate=None,
    verbose=10,
    early_stopping_rounds=5,
)
model_2.fit(X_train, Y_train, cat_features=categorical_cols)

### Evaluation_2

In [ ]:
Y_pred_prob = model_2.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(Y_test, Y_pred_prob)
Y_pred = model_2.predict(X_test)
report = classification_report(Y_test, Y_pred, output_dict=True)
df_report = pd.DataFrame(report).transpose()
df_report, roc_auc

## XGBoost Model 1 - 65% Missing Threshold & Train,Val,Test (i.l.o. Train,Test)

### Pre-Requisites

In [ ]:
str_filename = 'train.csv'
str_local_path = f'./input/{str_filename}'
df = pd.read_csv(str_local_path)

### Pre-Processing

In [ ]:
#missingvals
df = df.dropna(subset=['TARGET'])
missing_vals = df.isnull().mean() * 100
missing_cols = missing_vals[missing_vals > 0]
missing_cols = missing_cols.sort_values(ascending=False)

#remove features with a missing rate threshold of 65%
cols_to_remove = missing_cols[missing_cols > 65].index.tolist()
df_clean = df.drop(columns=cols_to_remove)

#df for num cols and cat cols
numerical_cols = df_clean.select_dtypes(include=['float64', 'int64']).columns.drop('TARGET')
categorical_cols = df_clean.select_dtypes(include=['object']).columns

#impute on num and cat cols
df_clean[numerical_cols] = df_clean[numerical_cols].fillna(df_clean[numerical_cols].median())
for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

### Label Encoding Cat Cols

In [ ]:
label_encoders = {}
for column in df_clean.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    df_clean[column] = le.fit_transform(df_clean[column])
    label_encoders[column] = le

### Feature Scaling

In [ ]:
scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])

### Data Splitting

In [ ]:
# split the data into features and target
X = df_clean.drop(columns=['TARGET'])
Y = df_clean['TARGET']

# split the data into training (60%) and remaining (40%) sets
X_train, X_rem, Y_train, Y_rem = train_test_split(X, Y, test_size=0.4, random_state=42)

# split the remaining data into validation (50% of remaining i.e., 20% of total) and test sets (50% of remaining i.e., 20% of total)
X_val, X_test, Y_val, Y_test = train_test_split(X_rem, Y_rem, test_size=0.5, random_state=42)

### Train & Test

In [ ]:
# initialize the XGBoost model
model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')

# train the model on the training set
model.fit(X_train, Y_train, eval_set=[(X_val, Y_val)], early_stopping_rounds=10, verbose=True)

# make predictions on test set
Y_pred = model.predict(X_test)
Y_pred_proba = model.predict_proba(X_test)[:, 1]

### Evaluation

In [ ]:
# show
for i in range(10):
    print(f"Predicted: {Y_pred[i]}, Probability of default: {Y_pred_proba[i]:.4f}")

# calc accuracy
accuracy = accuracy_score(Y_test, Y_pred)
print(f"Accuracy: {accuracy:.4f}")

# calc ROC AUC score
roc_auc = roc_auc_score(Y_test, Y_pred_proba)
print(f"ROC AUC Score: {roc_auc:.4f}")